# CourtVision — Phase 3 Calibration Handover

This notebook records the current state of court calibration for handover to Claude. It is a handover/validation notebook, not a claim that full-clip calibration is solved. Run it in the existing Kaggle environment with the original Phase 3 notebook and SportsMOT frame files available.

## Project status

- **Phase 1 — Detection:** fine-tuned YOLO26n player detector. Previously used checkpoint: `/kaggle/input/datasets/dannkusi/courtvision-artifact/best.pt`.
- **Phase 2A–B — Tracking:** 300 frames from SportsMOT sequence `v_00HRwkvvjtQ_c007`. Optimized ByteTrack (`match_thresh=0.90`) reached IDF1 **47.89%**, MOTA **60.54%**, 76 ID switches. Baseline ByteTrack: IDF1 **36.45%**, MOTA **60.66%**, 79 switches.
- **Phase 2C — Tracklet reconciliation:** jersey-appearance clustering and conservative offline tracklet linking produced only a marginal IDF1 change (**47.97%**); optimized ByteTrack remains the practical baseline.
- **Phase 3 — Current:** court-region extraction, camera-motion estimation, dynamic court calibration, lane/baseline/free-throw geometry, then image-to-court mapping.

Frames are 1280×720 in the prior setup. The broadcast camera pans/reframes, so one static homography for all 300 frames is invalid.

## Important conclusions

### Static/global calibration failed
- Frame 100 vs frame 200 propagated anchor comparison: mean corner disagreement **229.28 px**, maximum **349.76 px**.
- Frame 150 comparison between anchors 100 and 200: mean disagreement **306.01 px**.
Do not use one global homography across this moving-camera clip.

### Best manual reference frames
The scoreboard hides the lower lane/free-throw corner in many frames. The user identified **frames 105 and 115** as the clearest usable manual annotation frames.

### Calibration evidence so far
- Court-only ORB pair: **1,182 matches**, **876 RANSAC inliers**, ratio **0.7411**.
- Independent vs propagated:
  - Frame 105: mean disagreement **9.4 px**, max **13.41 px**.
  - Frame 115: mean disagreement **62.92 px**, max **79.82 px**.
- Frame 115 automatic baseline corner reconstruction:
  - lower `[340.44, 540.66]`, manual difference **6.86 px**
  - upper `[435.97, 437.65]`, manual difference **4.95 px**
  - mean **5.91 px**
- Frame 115 automatic full-lane reconstruction:
  - `[[340.44, 540.66], [435.97, 437.65], [861.26, 482.80], [797.80, 592.70]]`
  - compared with manual lane corners: mean error **4.10 px**, max **6.86 px**.

The frame-115 lane reconstruction is the strongest current result, but it is a single-frame validation—not proof that the method generalizes to every frame.

## User-supplied manual landmarks

These are image pixel coordinates, in the exact order supplied. **Do not reorder or infer their semantic meaning** until checking the existing Phase 3 notebook's point correspondence convention.

Frame 105:
`[[280,471],[323,422],[372,370],[622,394],[373,482],[543,499],[732,518],[793,411]]`

Frame 115:
`[[365,522],[416,466],[531,449],[681,466],[520,561],[759,588],[799,592],[861,486]]`

An older experiment used:
`[[302,648],[457,480],[601,496],[1101,557],[471,675],[810,716],[1185,570],[1098,716]]`
Do not mix that older point set with the current frame-105/115 annotations.

In [ ]:
import numpy as np

LINE_POINTS = {
    105: np.array([[280,471],[323,422],[372,370],[622,394],
                   [373,482],[543,499],[732,518],[793,411]], dtype=np.float32),
    115: np.array([[365,522],[416,466],[531,449],[681,466],
                   [520,561],[759,588],[799,592],[861,486]], dtype=np.float32),
}
OLDER_EXPERIMENT_POINTS = np.array(
    [[302,648],[457,480],[601,496],[1101,557],
     [471,675],[810,716],[1185,570],[1098,716]], dtype=np.float32
)
for fid, pts in LINE_POINTS.items():
    print(f"Frame {fid}: {len(pts)} points; x={pts[:,0].min():.0f}..{pts[:,0].max():.0f}, "
          f"y={pts[:,1].min():.0f}..{pts[:,1].max():.0f}")

## Local ORB/RANSAC diagnostics, frames 106–115

| Frame | Inlier ratio | Mean motion (px) |
|---:|---:|---:|
| 106 | 0.8431 | 3.4366 |
| 107 | 0.7854 | 3.0844 |
| 108 | 0.7948 | 3.7554 |
| 109 | 0.7767 | 4.0756 |
| 110 | 0.7139 | 8.3885 |
| 111 | 0.8009 | 4.8404 |
| 112 | 0.8052 | 4.9224 |
| 113 | 0.8004 | 4.8288 |
| 114 | 0.8122 | 4.8286 |
| 115 | 0.7178 | 9.0020 |

An inlier ratio alone is not sufficient. Also inspect court-feature spatial distribution, motion magnitude, projected court corners, polygon validity, and visual overlays.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

motion_data = pd.DataFrame({
    "frame": [106,107,108,109,110,111,112,113,114,115],
    "inlier_ratio": [0.8431,0.7854,0.7948,0.7767,0.7139,0.8009,0.8052,0.8004,0.8122,0.7178],
    "mean_motion_px": [3.436643,3.084421,3.755364,4.075619,8.388524,
                       4.840406,4.922351,4.828831,4.828610,9.001953],
})
display(motion_data)
fig, ax = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
ax[0].plot(motion_data.frame, motion_data.inlier_ratio, marker="o")
ax[0].set_ylabel("RANSAC inlier ratio"); ax[0].grid(True)
ax[1].plot(motion_data.frame, motion_data.mean_motion_px, marker="o")
ax[1].set_ylabel("Mean motion (px)"); ax[1].set_xlabel("Frame"); ax[1].grid(True)
plt.tight_layout()

## Previously saved Kaggle artifacts

- `/kaggle/working/courtvision_calibration_anchors.json` — registered anchors for frames 100 and 200 (both reported `ready`); retain as historical diagnostics, not as one stable global calibration.
- `/kaggle/working/courtvision_anchor_comparison_150.png` — frame-150 comparison; large disagreement.
- `FRAME_PATHS` — previously defined list of frame paths in the working notebook.

The exact filenames of all frame-115 line-fit visualizations were not provided. Inspect the current Kaggle working directory and notebook outputs rather than assuming paths.

In [ ]:
from pathlib import Path
import json

for p in [
    Path("/kaggle/working/courtvision_calibration_anchors.json"),
    Path("/kaggle/working/courtvision_anchor_comparison_150.png"),
]:
    print(p, "exists:", p.exists(), "size:", p.stat().st_size if p.exists() else None)

p = Path("/kaggle/working/courtvision_calibration_anchors.json")
if p.exists():
    print(json.dumps(json.loads(p.read_text()), indent=2)[:12000])
else:
    print("Anchor JSON not found in this session; rerun the anchor-registration cell.")

## Instructions for Claude — inspect first, then continue

1. Open and inspect the actual current Phase 3 Kaggle notebook and artifacts. This handover does not contain every existing variable/function implementation.
2. Identify the code for court masking, court-only ORB, RANSAC homography, anchor propagation, line fitting, baseline reconstruction and lane-corner reconstruction.
3. Preserve the successful frame-115 lane reconstruction; reproduce the **4.10 px mean / 6.86 px maximum** manual-vs-automatic corner error before changing it.
4. Treat frames 105 and 115 as independently calibrated references. Verify the eight-point correspondence order in the original code.
5. Use short local propagation windows; re-anchor when confidence drops. Reject propagation across camera cuts or significant pan/zoom changes. Frame 115's 62.92 px independent-vs-propagated disagreement is a clear warning.
6. Add frame-level status (`independent`, `propagated`, `reconstructed`, `rejected`) and diagnostics. Require spatially distributed court-only inliers, valid convex/non-self-intersecting polygons, plausible corner ordering, and visual verification.
7. Canny/Hough on the full frame generated many false segments from spectators, players, text and broadcast graphics. Restrict line fitting to court regions and use basketball geometry constraints.
8. Once image-space calibration is stable, calculate image-to-court homographies and map player bottom-centres. State and verify the court coordinate convention and NCAA dimensions before hardcoding them.
9. Validate on frames beyond 105–115 and across other camera-motion segments. Do not claim full-clip calibration based on one successful frame.

### Non-negotiable cautions
- Scoreboard occlusion is a real limitation; do not fabricate hidden lane corners.
- Do not invent landmark coordinates or silently change their order.
- Do not use a static whole-clip homography.
- Do not equate a high RANSAC inlier ratio with correct court calibration.
- Keep existing Phase 1/2 work intact; make focused Phase 3 changes only.

## Next-step checklist

- [ ] Open the existing Phase 3 notebook and map its current cells/functions.
- [ ] Confirm frame indexing (`FRAME_PATHS[index]` versus numeric filename).
- [ ] Confirm the semantic correspondence/order of the eight manual landmarks.
- [ ] Reproduce frame-115 automatic lane corners and manual comparison.
- [ ] Reproduce frame-105 and frame-115 independent-vs-propagated comparison.
- [ ] Implement confidence-based rejection and re-anchoring.
- [ ] Validate another camera-motion segment and save overlays/diagnostic CSV.